# **SwipeWise — Part 5: Advanced Model Development & Tuning**
**Member 5 Assignment Work:** Tree Ensembles & Hyperparameter Optimization Schemes  

---
### **Why Ensembles are Required:**
As established in the previous **EDA and Feature Engineering** phases, individual features within the synthetic dating platform dataset exhibit a highly distributed, "weak signal" correlation pattern to the multi-class target labels (`match_outcome`). Standalone traditional linear frameworks run a severe risk of underfitting, while shallow decision boundaries often overfit to localized programmatic noise. 

To combat this challenge, **Member 5** introduces advanced non-linear ensemble frameworks: **Random Forest** (a parallel bagging technique) and **Gradient Boosting** (a sequential error-correcting boosting architecture). By deploying randomized grid cross-validation searches (`RandomizedSearchCV`), we can isolate structural settings that maximize accurate multivariate target separation without letting the underlying trees memorize dataset irregularities.

### Step 1 — Advanced Ensemble Initialization & Benchmarking
We synchronize our workspace with the preprocessed data splits (`X_train`, `y_train`) generated by the preprocessing team. We then initialize and fit two baseline advanced ensemble estimators with default, factory configurations to establish raw baseline scores.

In [ ]:
print("--- Step 1: Training Baseline Ensemble Estimators ---")

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

# 1. Load the preprocessed data splits generated by Part 3
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

print(f"[SUCCESS] Data synchronized. Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

# Initialize base models (n_jobs=-1 forces the system to use all available CPU cores for speed)
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
gb_base = GradientBoostingClassifier(random_state=42)

# Fit un-optimized base models
rf_base.fit(X_train, y_train)
gb_base.fit(X_train, y_train)

# Calculate baseline metrics
acc_rf_base = accuracy_score(y_test, rf_base.predict(X_test)) * 100
acc_gb_base = accuracy_score(y_test, gb_base.predict(X_test)) * 100

print(f"-> Base Random Forest Test Accuracy: {acc_rf_base:.2f}%")
print(f"-> Base Gradient Boosting Test Accuracy: {acc_gb_base:.2f}%")

### Step 2 — Hyperparameter Tuning for Random Forest
We apply `RandomizedSearchCV` across a Stratified 3-Fold cross-validation split to systematically optimize our Random Forest configuration. Tuning structural boundaries such as `max_depth` (tree depth limits) and `min_samples_leaf` forces the model to regularize. This keeps individual tree paths from memorizing programmatic data noise, allowing them to extract real multivariate behaviors over weak signals.

In [ ]:
print("--- Step 2: Hyperparameter Tuning for Random Forest ---")

# Hyperparameter distribution grid designed to find patterns without overfitting
rf_param_dist = {
    'n_estimators': [100, 200, 300],          # Total voting trees
    'max_depth': [15, 25, None],               # Structural tree depth limits
    'min_samples_split': [2, 5, 10],          # Min samples needed to branch out
    'min_samples_leaf': [1, 2, 4],            # Min sample boundary for leaf nodes
    'bootstrap': [True, False]                 # Resampling behaviors
}

# 3-Fold Stratified search (Matches the perfect target class balance confirmed in EDA)
rf_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_param_dist, 
    n_iter=8, 
    cv=3, 
    scoring='accuracy', 
    random_state=42, 
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
tuned_rf_model = rf_search.best_estimator_
acc_rf_tuned = accuracy_score(y_test, tuned_rf_model.predict(X_test)) * 100

print(f"✨ Optimized Random Forest Parameters: {rf_search.best_params_}")
print(f"-> Tuned Random Forest Test Accuracy: {acc_rf_tuned:.2f}%")

### Step 3 — Hyperparameter Tuning for Gradient Boosting
To tune our sequential Gradient Boosting architecture, we execute a randomized cross-validation grid to optimize its learning patterns. By controlling the `learning_rate` (contribution shrinkage of successive trees) and `subsample` (row sampling fraction per boosting stage), we regularize the step size. This ensures the model gradually minimizes loss errors without aggressively chasing synthetic background outliers.

In [ ]:
print("--- Step 3: Hyperparameter Tuning for Gradient Boosting ---")

# Shrinkage and regularized gradient distribution parameters
gb_param_dist = {
    'n_estimators': [100, 150, 200],          # Total boosting cycles
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Contribution step shrinkage factor
    'max_depth': [3, 5, 7],                   # Shallow tree boundaries to ignore noise
    'subsample': [0.8, 1.0]                   # Training sample fraction per step iteration
}

gb_search = RandomizedSearchCV(
    estimator=gb_base, 
    param_distributions=gb_param_dist, 
    n_iter=6, 
    cv=3, 
    scoring='accuracy', 
    random_state=42, 
    n_jobs=-1
)

gb_search.fit(X_train, y_train)
tuned_gb_model = gb_search.best_estimator_
acc_gb_tuned = accuracy_score(y_test, tuned_gb_model.predict(X_test)) * 100

print(f"✨ Optimized Gradient Boosting Parameters: {gb_search.best_params_}")
print(f"-> Tuned Gradient Boosting Test Accuracy: {acc_gb_tuned:.2f}%")

### Step 4 — Performance Metrics Visualization & Model Exportation
To evaluate our hyperparameter optimization gains, we compile our accuracy metrics and build a comparative bar graph mapping our base estimators against their optimized variants. After plotting the comparative results, the tuned models are serialized into binary formats (`.joblib`). This allows Member 6 to load them directly into the final section to generate classification tables and compare them against `auto-sklearn` benchmarks.

In [ ]:
print("--- Step 4: Generating Visual Comparison Graph & Exporting Models ---")

# Define plot inputs based on our calculations
models_labeled = ['Base Random Forest', 'Tuned Random Forest', 'Base Gradient Boosting', 'Tuned Gradient Boosting']
accuracies_recorded = [acc_rf_base, acc_rf_tuned, acc_gb_base, acc_gb_tuned]
bar_colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

# Create comparative bar chart plot
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(models_labeled, accuracies_recorded, color=bar_colors, width=0.45, alpha=0.9)

# Add accurate percentage tags directly on top of the bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2., 
        height + 0.6, 
        f"{height:.2f}%", 
        ha='center', va='bottom', fontsize=10, weight='bold'
    )

# Chart layout styling to match master notebook layout
ax.set_ylabel('Test Set Target Accuracy (%)', weight='bold', fontsize=11)
ax.set_title('Member 5: Predictive Accuracy Gains via Ensemble Hyperparameter Tuning', weight='bold', fontsize=13, pad=15)
ax.set_ylim(0, max(accuracies_recorded) + 12)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Handoff Serialization Phase — Saves the files safely to your folder
joblib.dump(tuned_rf_model, 'tuned_random_forest.joblib')
joblib.dump(tuned_gb_model, 'tuned_gradient_boosting.joblib')

print("\n[SUCCESS]: Model serialization completed. Binary files saved successfully.")
print("=== Part 5 Pipeline Complete. Passing artifacts to Member 6 for Evaluation! ===")